# Partial Differential Equations: Finite Differences and Relaxation

---
REFS:
- Franklin, *Computational Methods for Physics*
- Landau, Páez, Bordeianu, *Computational Physics*
- Boudreau et al., *Applied Computational Physics*
- Barba group, [Laplace equation notes](https://barbagroup.github.io/essential_skills_RRC/laplace/1/)
---

```{admonition} Learning goals
:class: tip
After this notebook you should be able to
1. derive finite-difference formulas from Taylor series and know their order of accuracy;
2. discretize the Laplace/Poisson equation with the five-point stencil;
3. solve it with the relaxation methods Jacobi, Gauss–Seidel and SOR;
4. check a numerical solution against an exact one and choose a sensible stopping tolerance;
5. handle Dirichlet, Neumann and internal (electrode) boundary conditions.
```

In [ ]:
# Setup: all imports and small helpers used in the notebook
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, Image, display

try:
    from numba import njit, prange
except ImportError:            # numba is optional: fall back to plain Python
    def njit(f=None, **kwargs):
        return f if f is not None else (lambda g: g)
    prange = range
    print("numba not found: the 'numba' functions will run as plain (slow) Python")

os.makedirs("fig", exist_ok=True)

# Shared helper to embed videos and web pages (utils/embed.py in the repository)
import sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "utils")))
from embed import embed

## Why partial differential equations?

Many physical fields depend on several variables at once, such as the temperature $T(x, y, z, t)$ in a metal plate, the electric potential $V(x, y, z)$ in a device, or the displacement $u(x, t)$ of a string. Their laws relate partial derivatives with respect to several variables, so they are **partial differential equations (PDEs)**. For interactive examples see [VisualPDE](https://visualpde.com/explore.html).

In [ ]:
embed("https://www.youtube.com/watch?v=ly4S0oi3Yz8")

| Phenomenon | PDE | Type | What you must specify |
|---|---|---|---|
| Electrostatics, steady heat | Laplace/Poisson $\nabla^2 V = -\rho/\epsilon_0$ | Elliptic | Boundary values on a closed boundary |
| Heat conduction, diffusion | $\partial_t T = D\nabla^2 T$ | Parabolic | Initial condition + boundary values |
| String, sound, light | $\partial_t^2 u = c^2\nabla^2 u$ | Hyperbolic | Initial $u$ and $\partial_t u$ + boundary values |
| Quantum mechanics | $i\hbar\,\partial_t\psi = -\frac{\hbar^2}{2m}\nabla^2\psi + U\psi$ | Parabolic-like | Initial condition + boundary values |
| [Incompressible fluids](https://paveldogreat.github.io/WebGL-Fluid-Simulation/) | Navier–Stokes | Nonlinear, mixed | Initial condition + boundary values |

```{note} Classification
A linear second-order PDE in two variables, $A u_{xx} + B u_{xy} + C u_{yy} + \dots = 0$, is **elliptic** if $B^2 - 4AC < 0$ (steady states, e.g. Laplace), **parabolic** if $B^2 - 4AC = 0$ (irreversible evolution, e.g. heat) and **hyperbolic** if $B^2 - 4AC > 0$ (waves). The type tells you which conditions make the problem well posed, and therefore which numerical strategy fits: elliptic problems are solved for the whole domain at once, while parabolic and hyperbolic ones are marched in time (next notebooks).
```

The Laplace equation describes, for instance, the shape of a soap film on a wire frame, and the final state of a heat or diffusion process:

In [ ]:
embed("https://www.youtube.com/shorts/deQAnSDPrSs", width="40%", aspect=(9, 16))

In [ ]:
embed("https://visualpde.com/sim/?preset=heatEquation1D")

More demos: [VisualPDE wave equation](https://visualpde.com/sim/?preset=waveEquation1D), [PhET wave on a string](https://phet.colorado.edu/sims/html/wave-on-a-string/latest/wave-on-a-string_all.html), [Falstad ripple tank](https://falstad.com/ripple/), [acoustics animations](https://www.acs.psu.edu/drussell/demos.html).

Only simple geometries have analytical solutions (separation of variables, Fourier series, Green's functions). Irregular shapes, nonlinear materials or coupled fields require numerical methods. The simplest family follows three steps:

1. replace the continuous domain by a **grid**;
2. replace derivatives by **finite differences**;
3. solve the resulting **algebraic equations**.

## Finite differences in a nutshell

You already met numerical derivatives in [Numerical differentiation](../03-Scipy-NumericalCalculus/NumericalDifferentiation.ipynb). Here is the short version. On a grid $x_i = x_0 + i h$, write $u_i = u(x_i)$ and Taylor-expand:

$$
u_{i\pm1} = u_i \pm h\,u'_i + \frac{h^2}{2}u''_i \pm \frac{h^3}{6}u'''_i + \frac{h^4}{24}u''''_i \pm \dots
$$

Combining these expansions gives

| Formula | Expression | Error |
|---|---|---|
| Forward first derivative | $u'_i \approx \dfrac{u_{i+1} - u_i}{h}$ | $O(h)$ |
| Central first derivative | $u'_i \approx \dfrac{u_{i+1} - u_{i-1}}{2h}$ | $O(h^2)$ |
| Central second derivative | $u''_i \approx \dfrac{u_{i+1} - 2u_i + u_{i-1}}{h^2}$ | $O(h^2)$ |

For example, adding the expansions of $u_{i+1}$ and $u_{i-1}$ cancels the odd terms: $u_{i+1} + u_{i-1} = 2u_i + h^2 u''_i + \frac{h^4}{12}u''''_i + \dots$, which gives the second-derivative formula with error $\frac{h^2}{12}u''''_i$.

The plot below checks the orders for $u = \sin x$ at $x = 1$. Halving $h$ halves the forward error and divides the central errors by 4, until round-off ($\sim \varepsilon/h$ or $\varepsilon/h^2$) takes over for very small $h$.

In [ ]:
x0 = 1.0
h = np.logspace(-7, -0.5, 60)
err_fwd = np.abs((np.sin(x0 + h) - np.sin(x0)) / h - np.cos(x0))
err_cen = np.abs((np.sin(x0 + h) - np.sin(x0 - h)) / (2*h) - np.cos(x0))
err_2nd = np.abs((np.sin(x0 + h) - 2*np.sin(x0) + np.sin(x0 - h)) / h**2 + np.sin(x0))

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(h, err_fwd, "o-", ms=3, label="forward $u'$")
ax.loglog(h, err_cen, "s-", ms=3, label="central $u'$")
ax.loglog(h, err_2nd, "^-", ms=3, label="central $u''$")
ax.loglog(h, 0.3*h, "k--", lw=1, label=r"$\propto h$")
ax.loglog(h, 0.1*h**2, "k:", lw=1, label=r"$\propto h^2$")
ax.set(xlabel="$h$", ylabel="absolute error", title=r"Finite-difference errors for $\sin x$ at $x=1$", ylim=(1e-12, 1))
ax.legend(fontsize=8)
plt.show()

## From one to two dimensions

### One dimension: relaxation as "average of the neighbours"

Take $u''(x) = 0$ on $[0, 1]$ with $u(0) = 0$, $u(1) = 1$ (exact solution $u = x$). The central difference turns it into one equation per interior point:

$$
\frac{u_{i+1} - 2u_i + u_{i-1}}{h^2} = 0 \quad\Longleftrightarrow\quad u_i = \frac{u_{i+1} + u_{i-1}}{2}.
$$

There are two ways to read this:
- **As a linear system.** The equations form a tridiagonal matrix system $A\,u = b$, exactly as in the [finite-difference BVP](../06-ODE-BVP/BVP.ipynb) notebook. It can be solved directly.
- **As a fixed-point rule.** Each value is the average of its neighbours. Start from any guess and sweep the grid repeatedly, replacing each value by that average, until nothing changes. This is **relaxation**.

The plot shows the relaxation sweeps: the information from the boundaries slowly diffuses inwards. In fact, each sweep behaves like one time step of the heat equation, and the final state is its steady state.

In [ ]:
N1 = 21
u = np.zeros(N1)
u[-1] = 1.0                                   # boundary values u(0)=0, u(1)=1
x1 = np.linspace(0, 1, N1)
fig, ax = plt.subplots(figsize=(6, 4))
sweep = 0
for target in [0, 10, 50, 200, 1000]:
    while sweep < target:
        u[1:-1] = 0.5*(u[2:] + u[:-2])        # one relaxation sweep (Jacobi)
        sweep += 1
    ax.plot(x1, u, "o-", ms=3, label=f"{sweep} sweeps")
ax.plot(x1, x1, "k--", lw=1, label="exact $u=x$")
ax.set(xlabel="$x$", ylabel="$u$", title="1D relaxation for $u''=0$")
ax.legend()
plt.show()

### Two dimensions: the five-point stencil

The electrostatic potential obeys the Poisson equation (the Laplace equation when $\rho = 0$):

$$
\nabla^2 V = \frac{\partial^2 V}{\partial x^2} + \frac{\partial^2 V}{\partial y^2} = -\frac{\rho}{\epsilon_0}.
$$

On a square grid $x_i = ih$, $y_j = jh$, we write $V_{i,j} = V(x_i, y_j)$ and apply the central second difference in each direction:

$$
\frac{V_{i+1,j} - 2V_{i,j} + V_{i-1,j}}{h^2} + \frac{V_{i,j+1} - 2V_{i,j} + V_{i,j-1}}{h^2} = -\frac{\rho_{i,j}}{\epsilon_0}.
$$

Solving for $V_{i,j}$ gives

$$
\boxed{V_{i,j} = \frac{1}{4}\left(V_{i+1,j} + V_{i-1,j} + V_{i,j+1} + V_{i,j-1} + \frac{\rho_{i,j}h^2}{\epsilon_0}\right)}
$$

So the potential at each point is the average of its four neighbours plus a source term (figure from the Barba group):

```{figure} https://barbagroup.github.io/essential_skills_RRC/laplace/figures/laplace.svg
:width: 45%
:align: center
The five-point stencil.
```

As in 1D, the problem has two readings:
1. **Sparse linear system.** There is one equation per interior point, so $(N-2)^2$ unknowns. Each row has at most 5 non-zeros.
2. **Relaxation.** Sweep the grid repeatedly, applying the boxed formula, until the values stop changing. Relaxation is simple, uses little memory and adapts easily to complicated boundaries.

## The model problem

We solve the Laplace equation on the unit square with Dirichlet boundary conditions:

$$
V(x, 0) = \sin(\pi x), \qquad V(x, 1) = V(0, y) = V(1, y) = 0.
$$

Separation of variables gives the exact solution

$$
V(x, y) = \sin(\pi x)\,\frac{\sinh\big(\pi(1 - y)\big)}{\sinh \pi},
$$

which we use to measure errors.

```{admonition} Conventions used in all the code
:class: note
- The grid has `N` points per side, including the boundaries: `x = np.linspace(0, 1, N)` and `h = 1/(N-1)`.
- `V[i, j]` is $V(x_i, y_j)$: the first index is $x$ and the second is $y$. That is why plots use `V.T` with `origin="lower"`.
- Boundary rows and columns (`i` or `j` equal to `0` or `N-1`) are never updated by the relaxation step.
```

In [ ]:
def grid(N):
    """Grid of N x N points on the unit square. XX[i, j] = x_i, YY[i, j] = y_j."""
    x = np.linspace(0.0, 1.0, N)
    h = x[1] - x[0]
    XX, YY = np.meshgrid(x, x, indexing="ij")
    return x, h, XX, YY


def bc(V):
    """Dirichlet boundary conditions of the model problem (modifies V in place)."""
    x, h, XX, YY = grid(V.shape[0])
    V[:, 0] = np.sin(np.pi * x)   # bottom, y = 0
    V[:, -1] = 0.0                # top,    y = 1
    V[0, :] = 0.0                 # left,   x = 0
    V[-1, :] = 0.0                # right,  x = 1


def exact(N):
    """Exact solution of the model problem on the N x N grid."""
    x, h, XX, YY = grid(N)
    return np.sin(np.pi*XX) * np.sinh(np.pi*(1 - YY)) / np.sinh(np.pi)


def show(V, ax=None, title="", levels=10, cmap="viridis", label="$V$"):
    """2D map of V with contour lines."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(4.5, 3.8))
    x = np.linspace(0, 1, V.shape[0])
    im = ax.imshow(V.T, origin="lower", extent=(0, 1, 0, 1), cmap=cmap)
    if V.max() > V.min():
        lev = np.linspace(V.min(), V.max(), levels + 2)[1:-1]    # skip the extreme values
        ax.contour(x, x, V.T, levels=lev, colors="w", linewidths=0.6)
    plt.colorbar(im, ax=ax, label=label)
    ax.set(xlabel="$x$", ylabel="$y$", title=title)
    return ax


def show3d(V, ax=None, title="", cmap="viridis"):
    """3D surface of V."""
    if ax is None:
        ax = plt.figure(figsize=(5, 4)).add_subplot(projection="3d")
    x, h, XX, YY = grid(V.shape[0])
    ax.plot_surface(XX, YY, V, cmap=cmap, edgecolor="none")
    ax.set(xlabel="$x$", ylabel="$y$", zlabel="$V$", title=title)
    return ax


N = 41
V = np.zeros((N, N))
bc(V)
fig = plt.figure(figsize=(10, 3.8))
show(V, fig.add_subplot(121), "Boundary conditions")
show(exact(N), fig.add_subplot(122), "Exact solution")
plt.tight_layout()
plt.show()

## Relaxation methods

### Jacobi

The **Jacobi** method computes *all* new values from the *old* ones. In NumPy this is a single vectorized line. The right-hand side is evaluated completely before it is assigned, so only old values are used. The source term is passed as `rhs` $= \rho h^2/\epsilon_0$ (zero for Laplace).

In [ ]:
def jacobi(V, rhs):
    """One Jacobi sweep over the interior points (in place)."""
    V[1:-1, 1:-1] = 0.25*(V[2:, 1:-1] + V[:-2, 1:-1] + V[1:-1, 2:] + V[1:-1, :-2] + rhs[1:-1, 1:-1])

A generic driver repeats the sweeps until the largest change in one sweep is smaller than `tol`. It accepts any `step(V, rhs)` function and any `apply_bc(V)` function. `apply_bc` is called after every sweep, so it can also hold internal electrodes or derivative (Neumann) conditions.

In [ ]:
def solve(V, step, apply_bc, rhs=None, tol=1e-6, maxiter=100_000, nsave=0):
    """Relax V in place until max|change| < tol.

    Returns the number of sweeps, the history of max|change| and, if nsave > 0,
    a list of (sweep, copy of V) every nsave sweeps (useful for animations)."""
    if rhs is None:
        rhs = np.zeros_like(V)
    apply_bc(V)
    history, frames = [], [(0, V.copy())] if nsave else []
    for n in range(1, maxiter + 1):
        V_old = V.copy()
        step(V, rhs)
        apply_bc(V)
        history.append(np.max(np.abs(V - V_old)))
        if nsave and n % nsave == 0:
            frames.append((n, V.copy()))
        if history[-1] < tol:
            break
    return n, np.array(history), frames


N = 41
V = np.zeros((N, N))
n, hist, _ = solve(V, jacobi, bc, tol=1e-6)
print(f"Jacobi converged in {n} sweeps; max error vs exact = {np.max(np.abs(V - exact(N))):.2e}")

Snapshots of the relaxation from $V = 0$: the boundary information propagates about one cell per sweep, so the interior needs many sweeps.

In [ ]:
N = 41
V = np.zeros((N, N))
bc(V)
rhs = np.zeros_like(V)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
sweep = 0
for ax, target in zip(axes, [1, 30, 300, 3000]):
    while sweep < target:
        jacobi(V, rhs)
        sweep += 1
    show(V, ax, f"after {sweep} sweeps")
plt.tight_layout()
plt.show()

### Gauss–Seidel

The **Gauss–Seidel** method sweeps the grid point by point and overwrites the values *in place*. When it updates $V_{i,j}$, the neighbours $V_{i-1,j}$ and $V_{i,j-1}$ already hold their new values. Using fresher information makes it converge about twice as fast as Jacobi, and it needs no copy of the matrix. It cannot be written as a single NumPy slice, so we write explicit loops.

:::{admonition} Why `@njit`? A first look at numba
:class: note
Plain Python loops over a $100\times100$ grid take milliseconds per sweep, and relaxation needs thousands of sweeps. [Numba](https://numba.pydata.org/) is a *just-in-time (JIT) compiler*: the decorator `@njit` translates the function into machine code the **first time it is called**, so later calls run at C-like speed (about 100 times faster here).
- The first call is slower, because it includes the compilation.
- Inside the function use only NumPy arrays, numbers and plain loops (no lists of objects, pandas, plotting...).
- If numba is not installed, the setup cell replaces `@njit` by a do-nothing decorator, so the code still runs (slowly).

The section {ref}`fdr-performance` explains how to measure performance and goes deeper into numba, parallel loops and GPUs.
:::

:::{warning}
A double loop that writes into the same matrix it reads from is **Gauss–Seidel**, not Jacobi, even if the formula is identical. Keep the boundary points out of the loop.
:::

```{exercise} Gauss–Seidel sweep
:label: ex-fdr-gauss-seidel
Complete `gauss_seidel(V, rhs)`: loop over the **interior** points only (`i, j` from `1` to `N-2`) and overwrite `V[i, j]` with the boxed stencil formula, including `rhs[i, j]`. That is about four lines. Then run the next cell, which compares the number of sweeps with Jacobi.
```

In [ ]:
@njit
def gauss_seidel(V, rhs):
    """One Gauss-Seidel sweep over the interior points (in place)."""
    N = V.shape[0]
    # YOUR CODE HERE
    pass

In [ ]:
N = 41
for name, step in [("Jacobi", jacobi), ("Gauss-Seidel", gauss_seidel)]:
    V = np.zeros((N, N))
    n, hist, _ = solve(V, step, bc, tol=1e-6)
    print(f"{name:13s}: {n:5d} sweeps, max error vs exact = {np.max(np.abs(V - exact(N))):.2e}")

## Is it correct? Validation and stopping

### The order of the method

Two different errors are mixed in a relaxation result:
- the **discretization error**: the five-point equations are only an approximation of Laplace's equation, with error $O(h^2)$;
- the **iteration error**: we stop the sweeps before the five-point equations are satisfied exactly.

Our model problem has an exact solution, `exact(N)`, so we can measure both. First we remove the iteration error by converging very tightly (`tol=1e-10`). What remains is the discretization error, and it should decrease like $h^2$.

In [ ]:
Ns = np.array([11, 21, 41, 61, 81])
errors = []
for N in Ns:
    V = np.zeros((N, N))
    n, _, _ = solve(V, jacobi, bc, tol=1e-10)
    errors.append(np.max(np.abs(V - exact(N))))
    print(f"N = {N:3d}: {n:6d} sweeps, max error vs exact = {errors[-1]:.2e}")
hs = 1.0/(Ns - 1)
slope = np.polyfit(np.log(hs), np.log(errors), 1)[0]

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.loglog(hs, errors, "o-", label=f"five-point stencil (slope {slope:.2f})")
ax.loglog(hs, errors[0]*(hs/hs[0])**2, "k--", lw=1, label=r"$\propto h^2$")
ax.set(xlabel="$h$", ylabel=r"$\max|V - V_\mathrm{exact}|$", title="Discretization error (converged Jacobi)")
ax.set_xticks(hs, [f"1/{N-1}" for N in Ns])
ax.minorticks_off()
ax.legend()
plt.show()

```{exercise} Order of accuracy
:label: ex-fdr-order
1. What is the slope of the error plot above, and where does it come from?
2. If you need a maximum error of $10^{-6}$ for this problem, roughly what grid size $N$ do you need? How many Jacobi sweeps would that take?

```

```{admonition} Solving the linear system directly
:class: tip
The five-point equations for all interior points form a linear system $A\,\mathbf{v} = \mathbf{b}$ with about 5 non-zero entries per row (a *sparse* matrix). Instead of relaxing, it can be solved in one shot with a sparse direct solver such as `scipy.sparse.linalg.spsolve`. We build such a matrix ourselves for the implicit methods of the [heat equation](PDE-Parabolic-Heat.ipynb), and in [PDELibs](PDELibs.ipynb) the `findiff` library builds and solves it for us.
```

### A small change per sweep is not a small error

The driver stops when the largest change in one sweep is below `tol`. But relaxation converges slowly: each Jacobi sweep reduces the iteration error only by the factor $\rho \approx 1 - \frac{\pi^2 h^2}{2}$ (the *spectral radius*). So the remaining iteration error is roughly $\text{change}/(1 - \rho)$, which can be much larger than the change itself.

The plot below follows Jacobi for $N = 81$, comparing at every sweep the change (what the code monitors) with the error against the exact solution (what we really care about). The error cannot go below the discretization error of this grid (dashed line): sweeping further than that is wasted work. (The short dip below the dashed line near sweep 11000 is a coincidence: there the iteration and discretization errors have opposite signs and partly cancel.)

In [ ]:
N = 81
V = np.zeros((N, N))
bc(V)
rhs = np.zeros_like(V)
V_exact = exact(N)
changes, errs = [], []
for sweep in range(20000):
    V_old = V.copy()
    jacobi(V, rhs)
    changes.append(np.max(np.abs(V - V_old)))
    errs.append(np.max(np.abs(V - V_exact)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(changes, label="max change per sweep (what we monitor)")
ax.semilogy(errs, label="max error vs exact (what we want)")
ax.axhline(errs[-1], color="k", ls="--", lw=1, label="discretization error of this grid")
for tol in [1e-4, 1e-6]:
    k = np.argmax(np.array(changes) < tol)
    ax.axvline(k, color="gray", ls=":", lw=1)
    ax.text(k, 0.3, f" tol={tol:g}\n error={errs[k]:.1e}", fontsize=8)
ax.set(xlabel="sweep", ylabel="value", title=f"Jacobi, N={N}")
ax.legend(loc="lower left", fontsize=8)
plt.show()

```{exercise} Choosing the tolerance
:label: ex-fdr-tolerance
Using the plot above:
1. How large is the error when the code stops with `tol = 1e-4`? Why is it so much larger than `tol`?
2. Roughly which `tol` would you choose for $N = 81$? How does the number of sweeps needed by Jacobi grow with $N$, and what does that imply for the tolerance on finer grids?

```

## Successive over-relaxation (SOR)

Gauss–Seidel moves each value *towards* the neighbour average. SOR moves it *further* in that direction:

$$
V_{i,j} \leftarrow (1 - \omega)\,V_{i,j} + \omega\, V^{GS}_{i,j}, \qquad 1 < \omega < 2,
$$

where $V^{GS}_{i,j}$ is the Gauss–Seidel value (the boxed stencil). With $\omega = 1$ it is Gauss–Seidel again. For the Laplace equation on a square with $N$ points per side, the optimal value is

$$
\omega_\mathrm{opt} = \frac{2}{1 + \sin(\pi h)}, \qquad h = \frac{1}{N - 1}.
$$

At this value the number of sweeps grows like $N$ instead of $N^2$.

```{warning}
Over-relaxation must be applied *inside* the in-place Gauss–Seidel loop. Over-relaxing Jacobi (mixing a whole Jacobi sweep with the old matrix) diverges for $\omega > 1$.
```

```{exercise} SOR
:label: ex-fdr-sor
1. Complete `sor(V, rhs, omega)`: copy your Gauss–Seidel loop and change the update line to the SOR formula.
2. Run the next (given) cell. It counts the sweeps needed to reach `tol=1e-6` for $N = 41$ and several values of $\omega$, and compares Jacobi, Gauss–Seidel and SOR($\omega_\mathrm{opt}$) for $N = 11, 21, 41, 81$. Is the best $\omega$ close to $\omega_\mathrm{opt}$? How does the number of sweeps of each method grow with $N$?
```

In [ ]:
@njit
def sor(V, rhs, omega):
    """One SOR sweep over the interior points (in place)."""
    N = V.shape[0]
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing sor (and gauss_seidel) above
def omega_opt(N):
    return 2.0/(1.0 + np.sin(np.pi/(N - 1)))

N = 41
omegas = np.arange(1.0, 2.0, 0.05)
sweeps = []
for omega in omegas:
    V = np.zeros((N, N))
    n, _, _ = solve(V, lambda V, r: sor(V, r, omega), bc, tol=1e-6)
    sweeps.append(n)

Ns = [11, 21, 41, 81]
counts = {"Jacobi": [], "Gauss-Seidel": [], "SOR($\\omega_\\mathrm{opt}$)": []}
for Ni in Ns:
    w = omega_opt(Ni)
    steps = [jacobi, gauss_seidel, lambda V, r: sor(V, r, w)]
    for (name, c), step in zip(counts.items(), steps):
        V = np.zeros((Ni, Ni))
        c.append(solve(V, step, bc, tol=1e-6)[0])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.semilogy(omegas, sweeps, "o-")
ax1.axvline(omega_opt(N), color="r", ls="--", label=rf"$\omega_\mathrm{{opt}}={omega_opt(N):.3f}$")
ax1.set(xlabel=r"$\omega$", ylabel="sweeps to tol=1e-6", title=f"SOR, N={N}")
ax1.legend()
for name, c in counts.items():
    ax2.loglog(Ns, c, "o-", label=name)
Ns_arr = np.array(Ns)
ax2.loglog(Ns_arr, 3*Ns_arr**2, "k--", lw=1, label=r"$\propto N^2$")
ax2.loglog(Ns_arr, 3*Ns_arr, "k:", lw=1, label=r"$\propto N$")
ax2.set(xlabel="$N$", ylabel="sweeps to tol=1e-6", title="Cost of the relaxation methods")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

(fdr-performance)=
## Performance: measure, vectorize, compile, parallelize

Relaxation needs thousands of sweeps over $N^2$ points, so speed matters. The rule is: **first measure, then optimize the part that actually takes the time**, and remember that a better algorithm usually beats faster hardware.

### Measure first

The simplest tools:
- `time.perf_counter()` around a block of code (used below);
- the IPython magic `%timeit statement`, which repeats a statement and reports mean and spread;
- a **profiler**, which tells you *where* the time goes inside a function. `cProfile` comes with Python (`%prun statement` is its IPython shortcut).

For line-by-line timing install [line_profiler](https://github.com/pyutils/line_profiler) (`pip install line_profiler`) and use `%load_ext line_profiler` and then `%lprun -f solve solve(V, jacobi, bc)`. See also [timing and profiling in Jupyter](https://jakevdp.github.io/PythonDataScienceHandbook/01.07-timing-and-profiling.html).

In [ ]:
N = 100
V = np.zeros((N, N))
bc(V)
rhs = np.zeros_like(V)
%timeit -n 200 -r 5 jacobi(V, rhs)

In [ ]:
import cProfile
import pstats

N = 81
V = np.zeros((N, N))
profiler = cProfile.Profile()
profiler.enable()
solve(V, jacobi, bc, tol=1e-6)
profiler.disable()
pstats.Stats(profiler).strip_dirs().sort_stats("tottime").print_stats(8)

Read the `ncalls` and `tottime` columns (number of calls, and time spent inside each function itself). The Jacobi sweep is the largest cost, as expected, but the profile also reveals something we did not expect: `linspace`, `_broadcast_to` and most of the `copy` calls come from `bc`. It calls `grid(N)` (with `linspace` and `meshgrid`) in **every sweep**, only to set boundary values that never change; its `cumtime` (time including the functions it calls) is almost as large as that of the sweep itself. The remaining `copy` is `solve` keeping the old matrix to measure the change.

Once you know it, the fixes are easy: apply fixed Dirichlet conditions only once (call `bc(V)` and pass `apply_bc=lambda V: None` to `solve`), and check convergence only every 10 or 50 sweeps. Without the profiler you would probably have tried to optimize the sweep itself. (Lines such as `select.kqueue` belong to Jupyter itself and can be ignored.)

### Vectorize or compile

We have three versions of the same sweep: Python loops, NumPy slicing and numba-compiled loops. The pure-Python version is obtained from the numba one with `.py_func`.

In [ ]:
N = 100
V = np.zeros((N, N))
bc(V)
rhs = np.zeros_like(V)
gs_python = getattr(gauss_seidel, "py_func", gauss_seidel)    # the same code without numba
for name, f, reps in [("Gauss-Seidel, Python loops", gs_python, 5),
                      ("Jacobi, NumPy slicing", jacobi, 200),
                      ("Gauss-Seidel, numba", gauss_seidel, 200)]:
    f(V, rhs)                                                   # warm-up (numba compiles here)
    t0 = time.perf_counter()
    for _ in range(reps):
        f(V, rhs)
    print(f"{name:28s}: {1e3*(time.perf_counter() - t0)/reps:8.3f} ms per sweep (N={N})")

### Numba in more detail

`@njit` ("no-Python just-in-time") compiles a function the first time it is called with a given combination of argument *types* (for example, a 2D `float64` array). Calling it later with another type triggers a new compilation. Useful options:

| Option | Effect |
|---|---|
| `@njit` | compile to machine code; error if something cannot be compiled |
| `@njit(cache=True)` | save the compiled code on disk, so the next session starts fast |
| `@njit(fastmath=True)` | allow reordering floating-point operations (faster, last digits may change) |
| `@njit(parallel=True)` + `prange` | split a loop among the CPU cores |

The cell below shows the compilation cost with a new function, the **residual** $\max|V_{i+1,j}+V_{i-1,j}+V_{i,j+1}+V_{i,j-1}-4V_{i,j}+\text{rhs}_{i,j}|$, which measures how well $V$ satisfies the discrete equations (a better stopping criterion than the change per sweep).

In [ ]:
@njit
def residual(V, rhs):
    """max |five-point equation residual| over the interior points."""
    N = V.shape[0]
    res = 0.0
    for i in range(1, N - 1):
        for j in range(1, N - 1):
            r = V[i+1, j] + V[i-1, j] + V[i, j+1] + V[i, j-1] - 4.0*V[i, j] + rhs[i, j]
            res = max(res, abs(r))
    return res

def residual_numpy(V, rhs):
    r = V[2:, 1:-1] + V[:-2, 1:-1] + V[1:-1, 2:] + V[1:-1, :-2] - 4*V[1:-1, 1:-1] + rhs[1:-1, 1:-1]
    return np.max(np.abs(r))

N = 400
V = np.random.default_rng(0).random((N, N))
rhs = np.zeros_like(V)
for label in ["1st call (compiles)", "2nd call"]:
    t0 = time.perf_counter()
    residual(V, rhs)
    print(f"numba {label:20s}: {1e3*(time.perf_counter() - t0):8.3f} ms")
t0 = time.perf_counter()
residual_numpy(V, rhs)
print(f"numpy {'':20s}: {1e3*(time.perf_counter() - t0):8.3f} ms   (same value: {np.isclose(residual(V, rhs), residual_numpy(V, rhs))})")

**Parallel loops.** With `@njit(parallel=True)`, a loop written with `prange` instead of `range` is split among the CPU cores. This is only correct if the iterations are **independent**:
- Jacobi is safe: every new value is computed from the *old* matrix and written into a *new* one.
- Gauss–Seidel is **not**: row $i$ reads row $i-1$, which another core may be updating at the same time (a *race condition*), so the result would depend on the timing of the threads. The red-black ordering (see the further exercises) fixes this.

```{exercise} Parallel Jacobi with numba
:label: ex-fdr-parallel
1. Complete `jacobi_parallel(V, rhs)`: loop over the interior points with `prange` for `i` and `range` for `j`, and write the stencil average into `V_new[i, j]` (reading only `V`). The copy back to `V` is already written.
2. Run the next (given) cell. It checks that the result equals that of `jacobi` after 100 sweeps, and measures the time per sweep of `jacobi` (NumPy), `gauss_seidel` (numba) and `jacobi_parallel` for $N = 50, 100, \dots, 3200$. Which method wins for small and for large $N$? Why is the parallel version slower on small grids? (Hint: starting and synchronizing the threads has a fixed cost per sweep.)
```

In [ ]:
@njit(parallel=True)
def jacobi_parallel(V, rhs):
    """One Jacobi sweep, parallelized over the rows with prange."""
    N = V.shape[0]
    V_new = V.copy()
    # YOUR CODE HERE
    pass
    V[:, :] = V_new

In [ ]:
# Given: run it after completing jacobi_parallel (and gauss_seidel) above
import numba

N = 61
V1, V2 = np.zeros((N, N)), np.zeros((N, N))
bc(V1), bc(V2)
rhs = np.zeros_like(V1)
for _ in range(100):
    jacobi(V1, rhs)
    jacobi_parallel(V2, rhs)
print("same result as jacobi:", np.allclose(V1, V2), "| threads:", numba.get_num_threads())

def time_per_sweep(step, N, reps):
    V = np.zeros((N, N))
    bc(V)
    rhs = np.zeros_like(V)
    step(V, rhs)                       # warm-up / compilation
    t0 = time.perf_counter()
    for _ in range(reps):
        step(V, rhs)
    return (time.perf_counter() - t0)/reps

Ns = [50, 100, 200, 400, 800, 1600, 3200]
methods = {"Jacobi, NumPy": jacobi, "Gauss-Seidel, numba": gauss_seidel, "Jacobi, numba parallel": jacobi_parallel}
fig, ax = plt.subplots(figsize=(6, 4))
for name, step in methods.items():
    times = [time_per_sweep(step, Ni, max(5, 200000//Ni**2)) for Ni in Ns]
    ax.loglog(Ns, 1e3*np.array(times), "o-", label=name)
ax.loglog(Ns, 1e3*2e-9*np.array(Ns)**2, "k--", lw=1, label=r"$\propto N^2$")
ax.set(xlabel="$N$", ylabel="ms per sweep", title=f"Time per sweep ({numba.get_num_threads()} threads)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### GPUs

A GPU has thousands of simple cores and a very fast memory. It pays off when
- the grid is **large** ($N \gtrsim 1000$): each kernel launch costs about $10\,\mu$s, so small grids are faster on the CPU;
- the data **stays on the GPU**: copying the matrix between CPU (host) and GPU (device) is slow, so copy once, do many sweeps, copy back once;
- you avoid moving scalars back every sweep (for example, check convergence only every 100 sweeps).

[CuPy](https://cupy.dev/) is a drop-in replacement for NumPy on NVIDIA GPUs: our `jacobi` function works **unchanged** with CuPy arrays, because it only uses slicing. [Numba](https://numba.readthedocs.io/en/stable/cuda/index.html) can also compile your own GPU *kernels*, where each GPU thread updates one grid point.

```{admonition} Running the GPU cells on Google Colab
:class: tip
Your laptop probably has no NVIDIA GPU (and the book is built without one), so these cells are not executed here. To try them:
1. open the notebook in [Google Colab](https://colab.research.google.com/) and choose *Runtime → Change runtime type → T4 GPU*;
2. check the GPU with `!nvidia-smi` (CuPy and numba come pre-installed);
3. copy the `jacobi` function and run the cells below. Compare the time per sweep with the CPU for $N = 100$ and $N = 4000$.

Alternatives with the same NumPy-like style are [JAX](https://jax.readthedocs.io/) (`jax.numpy` plus `jax.jit`) and [PyTorch](https://pytorch.org/) tensors, which also run on Apple GPUs with `device="mps"`.
```

In [ ]:
# GPU with CuPy: the NumPy-style jacobi(V, rhs) runs unchanged on GPU arrays
import cupy as cp

N = 4000
V = np.zeros((N, N))
bc(V)
V_gpu = cp.asarray(V)                 # copy host -> device (once)
rhs_gpu = cp.zeros_like(V_gpu)

jacobi(V_gpu, rhs_gpu)                # warm-up
cp.cuda.Stream.null.synchronize()     # GPU calls are asynchronous: wait before timing
t0 = time.perf_counter()
for _ in range(100):
    jacobi(V_gpu, rhs_gpu)
cp.cuda.Stream.null.synchronize()
print(f"CuPy: {1e3*(time.perf_counter() - t0)/100:.3f} ms per sweep (N={N})")
V = cp.asnumpy(V_gpu)                 # copy device -> host (once, at the end)

In [ ]:
# GPU with a numba.cuda kernel: one GPU thread per grid point
from numba import cuda

@cuda.jit
def jacobi_kernel(V, V_new, rhs):
    i, j = cuda.grid(2)               # global (i, j) index of this thread
    N = V.shape[0]
    if 0 < i < N - 1 and 0 < j < N - 1:
        V_new[i, j] = 0.25*(V[i+1, j] + V[i-1, j] + V[i, j+1] + V[i, j-1] + rhs[i, j])

N = 4000
V = np.zeros((N, N))
bc(V)
d_V, d_Vnew = cuda.to_device(V), cuda.to_device(V)     # host -> device
d_rhs = cuda.to_device(np.zeros_like(V))
threads = (16, 16)                                     # threads per block
blocks = ((N + 15)//16, (N + 15)//16)                  # enough blocks to cover the grid
for _ in range(1000):
    jacobi_kernel[blocks, threads](d_V, d_Vnew, d_rhs)
    d_V, d_Vnew = d_Vnew, d_V                          # swap: no copies needed
V = d_V.copy_to_host()                                 # device -> host

### The algorithm beats the hardware

Compilers and GPUs make each sweep faster by a constant factor (10 to 1000). Better algorithms change *how the cost grows with $N$*. To reduce the error by a factor $10^{-6}$ on a grid with $N = 1000$ points per side, Jacobi needs about $\frac{2N^2}{\pi^2}\ln 10^6 \approx 3\times10^6$ sweeps, while SOR with $\omega_\mathrm{opt}$ needs about $\frac{N}{2\pi}\ln 10^6 \approx 2000$, a factor of 1000. *Multigrid* methods need a number of sweeps independent of $N$. For moderate sizes, a sparse direct solver (see [PDELibs](PDELibs.ipynb)) is usually the fastest of all. Relaxation is still useful when memory is limited, the geometry is irregular, or as a building block of multigrid.

## Visualizing the relaxation

One helper animates the saved frames as a 2D map or a 3D surface.

In [ ]:
def animate(frames, kind="2d", interval=150):
    """Animate a list of (sweep, V) frames as a 2D map ('2d') or a 3D surface ('3d')."""
    fig = plt.figure(figsize=(5, 4))
    ax = fig.add_subplot(projection="3d" if kind == "3d" else None)
    x, h, XX, YY = grid(frames[0][1].shape[0])
    vmin = min(V.min() for _, V in frames)
    vmax = max(V.max() for _, V in frames)

    def update(k):
        ax.cla()
        sweep, V = frames[k]
        if kind == "3d":
            ax.plot_surface(XX, YY, V, cmap="viridis", vmin=vmin, vmax=vmax, edgecolor="none")
            ax.set_zlim(vmin, vmax)
        else:
            ax.imshow(V.T, origin="lower", extent=(0, 1, 0, 1), vmin=vmin, vmax=vmax)
        ax.set(xlabel="$x$", ylabel="$y$", title=f"sweep {sweep}")

    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval)
    plt.close(fig)
    return HTML(ani.to_jshtml())


N = 31
V = np.zeros((N, N))
n, hist, frames = solve(V, jacobi, bc, tol=1e-5, nsave=60)
animate(frames, "2d")

In [ ]:
animate(frames, "3d")

## Exercises: electrodes, charges and other boundaries

All these exercises reuse `solve`, `jacobi`, `show` and `show3d`. You only write a small boundary-condition function or a source array (a few lines); the code that solves and plots is given. We use `jacobi` as the step for problems with internal electrodes: it reads only old values, so values fixed by `apply_bc` stay consistent within each sweep.

In [ ]:
def grounded(V):
    """Set the four walls of the box to V = 0 (in place)."""
    V[0, :] = V[-1, :] = V[:, 0] = V[:, -1] = 0.0

```{exercise} Parallel-plate capacitor
:label: ex-fdr-capacitor
In a grounded box, add two plates at $x = 1/3$ (with $V = +3$) and $x = 2/3$ (with $V = -3$). Each plate has length $2/3$, centred in $y$.
1. Complete `bc_capacitor(V)`: set the two plates with slices such as `V[i1, j1:j2] = 3` (two lines).
2. Run the next cell, which solves with `N = 61` and plots the potential. The solution is kept in `V_cap` for the next exercise.
```

In [ ]:
def bc_capacitor(V):
    """Grounded box with plates at x = 1/3 (V = +3) and x = 2/3 (V = -3)."""
    N = V.shape[0]
    grounded(V)
    i1, i2 = N//3, 2*N//3            # x position of the plates
    j1, j2 = N//6, 5*N//6            # the plates cover the middle 2/3 in y
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing bc_capacitor
N = 61
V_cap = np.zeros((N, N))
n, _, _ = solve(V_cap, jacobi, bc_capacitor, tol=1e-5)
fig = plt.figure(figsize=(11, 4))
show(V_cap, fig.add_subplot(121), f"Capacitor ({n} sweeps)", cmap="RdBu_r")
show3d(V_cap, fig.add_subplot(122, projection="3d"), cmap="RdBu_r")
plt.tight_layout()
plt.show()

```{exercise} Electric field
:label: ex-fdr-efield
The next cell computes $\vec E = -\nabla V$ for the capacitor with `np.gradient` (axis 0 is $x$, axis 1 is $y$) and draws the field lines with `streamplot`. Where is the field strongest? Is it uniform between the plates? What happens near the edges of the plates?
```

In [ ]:
x, h, XX, YY = grid(V_cap.shape[0])
Ex, Ey = np.gradient(-V_cap, h)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
show(V_cap, ax, "Capacitor: potential and field lines", cmap="RdBu_r")
ax.streamplot(x, x, Ex.T, Ey.T, color="k", linewidth=0.7, density=1.2)
plt.tight_layout()
plt.show()

```{exercise} Charged ring
:label: ex-fdr-ring
In a grounded box, hold a circular ring of radius $1/4$, centred at $(1/2, 1/2)$, at $V = 3$.
1. Complete `bc_ring(V)`: build the boolean mask `ring = np.abs(np.hypot(XX - 0.5, YY - 0.5) - 0.25) < h` and set `V[ring] = 3` (two lines).
2. Run the next cell. What is the potential inside the ring, and why?

```

In [ ]:
def bc_ring(V):
    """Grounded box with a ring of radius 1/4 at V = 3."""
    x, h, XX, YY = grid(V.shape[0])
    grounded(V)
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing bc_ring
N = 61
V_ring = np.zeros((N, N))
n, _, _ = solve(V_ring, jacobi, bc_ring, tol=1e-5)
fig = plt.figure(figsize=(11, 4))
show(V_ring, fig.add_subplot(121), f"Ring at V=3 ({n} sweeps)")
show3d(V_ring, fig.add_subplot(122, projection="3d"))
plt.tight_layout()
plt.show()

```{exercise} Poisson equation: two square charges
:label: ex-fdr-charges
In a grounded box, place two square charge distributions of side $1/10$ on the diagonal: $\rho = +500$ centred at $(0.3, 0.3)$ and $\rho = -500$ centred at $(0.7, 0.7)$ (with $\epsilon_0 = 1$).
1. Complete `charge_density(XX, YY)` using boolean masks such as `(np.abs(XX - 0.3) < 0.05) & (np.abs(YY - 0.3) < 0.05)` (two lines).
2. Run the next cell, which solves $\nabla^2 V = -\rho$ with `rhs=rho*h**2` and plots $V$ with the field lines. Do the field lines go from the positive to the negative charge?
```

In [ ]:
def charge_density(XX, YY):
    """Two squares of side 1/10: rho = +500 around (0.3, 0.3) and -500 around (0.7, 0.7)."""
    rho = np.zeros_like(XX)
    # YOUR CODE HERE
    pass
    return rho

In [ ]:
# Given: run it after completing charge_density
N = 61
x, h, XX, YY = grid(N)
rho = charge_density(XX, YY)
V_rho = np.zeros((N, N))
n, _, _ = solve(V_rho, jacobi, grounded, rhs=rho*h**2, tol=1e-6)
Ex, Ey = np.gradient(-V_rho, h)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
show(V_rho, ax, f"Two square charges ({n} sweeps)", cmap="RdBu_r")
ax.streamplot(x, x, Ex.T, Ey.T, color="k", linewidth=0.6, density=1.0)
plt.tight_layout()
plt.show()

```{exercise} Insulated walls (Neumann condition): steady heat in a plate
:label: ex-fdr-neumann
The steady temperature in a plate also obeys $\nabla^2 T = 0$. Keep the left wall at $T = 100$ and the right wall at $T = 0$, and **insulate** the bottom and top walls: $\partial T/\partial y = 0$. A zero derivative at a wall means the wall value equals its neighbour (ghost-node idea, as in the BVP notebook): `T[:, 0] = T[:, 1]` and `T[:, -1] = T[:, -2]`.
1. Complete `bc_plate(T)` with these four conditions.
2. Run the next cell. It compares $T(x, y=1/2)$ with the exact solution $T = 100(1 - x)$, and also solves the case where only the lower half of the left wall is hot. Do the isotherms meet the insulated walls at right angles?
```

In [ ]:
def bc_plate(T):
    """Hot left wall, cold right wall, insulated bottom and top."""
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing bc_plate
def bc_plate_half(T):
    """As bc_plate, but only the lower half of the left wall is hot."""
    bc_plate(T)
    T[0, T.shape[1]//2:] = 0.0

N = 31
x, h, XX, YY = grid(N)
T1 = np.zeros((N, N))
solve(T1, jacobi, bc_plate, tol=1e-7)
T2 = np.zeros((N, N))
solve(T2, jacobi, bc_plate_half, tol=1e-7)

fig = plt.figure(figsize=(14, 4))
ax = fig.add_subplot(131)
ax.plot(x, T1[:, N//2], "o", ms=4, label="relaxation")
ax.plot(x, 100*(1 - x), "k-", lw=1, label="exact $100(1-x)$")
ax.set(xlabel="$x$", ylabel="$T(x, 1/2)$", title="Insulated top and bottom")
ax.legend()
show(T1, fig.add_subplot(132), "Hot left wall", cmap="inferno", label="$T$")
show(T2, fig.add_subplot(133), "Lower half of the left wall hot", cmap="inferno", label="$T$")
plt.tight_layout()
plt.show()

```{exercise} (Optional) 3D visualization with PyVista
:label: ex-fdr-pyvista
Install [PyVista](https://pyvista.org/) (`pip install pyvista trame`) and show the capacitor solution as a warped surface: build `pv.StructuredGrid(XX, YY, V_cap)`, store `V_cap` in it as point data (`grid["V"] = V_cap.ravel(order="F")`) and call `grid.plot(scalars="V")`.
```

In [ ]:
# YOUR CODE HERE
pass

## Summary

| Method | Work per sweep | Sweeps to converge | Notes |
|---|---|---|---|
| Jacobi | $O(N^2)$, vectorizes with NumPy | $\propto N^2$ | Simple, needs a copy, good for internal electrodes |
| Gauss–Seidel | $O(N^2)$, needs loops (numba) | $\approx \frac{1}{2}$ of Jacobi, $\propto N^2$ | In place, no copy |
| SOR ($\omega_\mathrm{opt}$) | same as Gauss–Seidel | $\propto N$ | Large gain from one extra parameter |
| Direct sparse solver (heat notebook, PDELibs) | one factorization | — | Exact discrete solution, fastest for moderate $N$ |

Key ideas:
- Finite differences come from Taylor series. The five-point stencil is $O(h^2)$ accurate, as the convergence plot shows.
- Relaxation reads the stencil as "each value is the average of its neighbours", so boundary information diffuses into the domain.
- A small change per sweep does not mean a small error. Check against an exact solution: iterating beyond the discretization error is wasted work, and on finer grids the tolerance must shrink (like $h^4$ for the change per sweep).
- Boundaries: Dirichlet fixes values. Neumann copies the neighbour value (ghost node). Electrodes are fixed values inside the domain, reapplied after each sweep.

## Further exercises
1. **Rectangular domain.** Generalize `grid`, `bc` and `jacobi` to $N_x \neq N_y$ and solve $L_x = 2$, $L_y = 1$.
2. **Red-black Gauss–Seidel.** Colour the grid like a chessboard. Updating all red points and then all black points is equivalent to Gauss–Seidel and can be vectorized with NumPy slices `V[1:-1:2, 1:-1:2]` and so on. Compare its speed with the numba version.
3. **L-shaped conductor.** Solve the Laplace equation in an L-shaped region (a square with one quadrant removed and held at $V = 0$). Where is the field strongest?
4. **Relaxation = diffusion.** Show that one Jacobi sweep equals one explicit step of the heat equation $\partial_t V = D\nabla^2 V$ with $D\,\Delta t/h^2 = 1/4$. This links to the next notebook.
5. **Multigrid (reading).** Find out why multigrid methods need a number of sweeps independent of $N$ (see Press et al., *Numerical Recipes*, ch. 20).